In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import scanpy as sc
from descope.tokenizer import tokenize_adata_to_hf_dataset_for_rna

In [3]:
def generate_condition_col_for_tahoe100m(adata_path: str) -> sc.AnnData:
    def _preprocess(x: str) -> str:
        import ast
        x: tuple = ast.literal_eval(x)[0]
        drug = x[0].strip()
        dose = float(x[1])
        norm_dose = float(dose / 5.0)  # max_dose in tahoe-100m: 5μM
        return f"{drug}_{norm_dose}"
    
    adata = sc.read_h5ad(adata_path)
    adata.obs["drug"] = adata.obs["drug"].str.strip()
    adata = adata[~adata.obs["drug"].isin(['Sacubitril/Valsartan', 'Verteporfin'])].copy()
    adata.obs["condition"] = adata.obs["drugname_drugconc"].apply(_preprocess)
    return adata

In [4]:
# raw counts
adata = generate_condition_col_for_tahoe100m("/fse/home/wupengpeng/perturbation_datasets/origin_datasets/Tahoe_100M/plate3_2k-obs.h5ad")

In [5]:
tokenize_adata_to_hf_dataset_for_rna(
    adata=adata,
    cell_line_col="cell_line",
    target_sum=1e4,
    pert_col="condition",
    ctrl_name="DMSO_TF_0.0",
    skip_raw_counts_check=True,
    save_dir="./tahoe100m_tokenized/plate3"
)

2026-07-20 11:11:53 | INFO | ------------------------- Preprocessing -------------------------
2026-07-20 11:11:53 | INFO | AnnData shape before preprocessing: (2000, 62710)
2026-07-20 11:11:53 | INFO | Remove cells with missing perturbation labels ...
2026-07-20 11:11:53 | INFO | No missing perturbation labels found.
2026-07-20 11:11:53 | INFO | Performing normalize_total (10000.0) and log1p ...
2026-07-20 11:11:53 | WARNING | Skipping raw counts check, directly applying normalization and log1p ...
2026-07-20 11:11:53 | INFO | AnnData shape after preprocessing: (2000, 62710)
2026-07-20 11:11:53 | INFO | ------------------------------ Done ------------------------------
2026-07-20 11:11:53 | INFO | Filtering perturbations ...
2026-07-20 11:11:53 | INFO | No perturbation filtering specified. Using all perturbations.
2026-07-20 11:11:53 | INFO | Tokenizing adata to huggingface dataset ...


Split Dataset Into Chunks:   0%|          | 0/1 [00:00<?, ?it/s]

Saving the dataset (0/2 shards):   0%|          | 0/2000 [00:00<?, ? examples/s]

Dataset({
    features: ['labels', 'pert_gene', 'celltype'],
    num_rows: 2000
})